In [1]:
import gradio as gr
from transformers import pipeline

In [2]:
print("Initializing Multi-Modal Prediction Engines...")

# 1. Text Pipeline: Sentiment Analysis using a compact DistilBERT architecture
text_engine = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

# 2. Vision Pipeline: Image Classification using a pre-trained Vision Transformer (ViT)
vision_engine = pipeline("image-classification", model="google/vit-base-patch16-224")

# 3. Audio Pipeline: Automatic Speech Recognition (ASR) using OpenAI's Whisper-Tiny
audio_engine = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

print("All engines successfully cached and loaded.")

Initializing Multi-Modal Prediction Engines...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] `ViTImageProcessor` requires torchvision (not installed); falling back to `ViTImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `ViTImageProcessorPil` directly to silence this warning.


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

All engines successfully cached and loaded.


In [3]:
# ------------------------------------------------------------------
# Wrapper Functions for Modality Processing
# ------------------------------------------------------------------
def process_text(input_text):
    if not input_text.strip():
        return "Please enter valid text."
    predictions = text_engine(input_text)[0]
    return f"Label: {predictions['label']} | Confidence: {predictions['score']:.4f}"

def process_image(input_image):
    if input_image is None:
        return "Please upload an image file."
    predictions = vision_engine(input_image)
    # Format top predictions into a readable clean readout
    output_str = ""
    for pred in predictions[:3]: # Grab top 3 predicted classes
        output_str += f"Class: {pred['label']} ({pred['score']*100:.2f}%)\n"
    return output_str

def process_audio(input_audio_path):
    if input_audio_path is None:
        return "Please upload or record an audio clip."
    # Whisper parses the audio file path and executes full frame conversion and decoding
    prediction = audio_engine(input_audio_path)
    return f"Transcribed Text:\n\"{prediction['text']}\""

In [4]:
# ------------------------------------------------------------------
# Building the Multi-Modal Dashboard using Gradio Blocks
# ------------------------------------------------------------------
with gr.Blocks(title="Multi-Modal AI Dashboard") as dashboard:
    gr.Markdown("# 🚀 Multi-Modal AI Production Dashboard")
    gr.Markdown("A unified interface featuring state-of-the-art models for Text, Vision, and Audio analysis.")
    
    # Create an elegant tabbed interface to separate our operational spaces
    with gr.Tabs():
        
        # --- TAB 1: TEXT ANALYSIS ---
        with gr.TabItem("📝 Text Sentiment Analysis"):
            gr.Markdown("### Analyze Text Sentiment")
            text_input = gr.Textbox(lines=4, placeholder="Type something expressive here...", label="Input String")
            text_button = gr.Button("Analyze Sentiment", variant="primary")
            text_output = gr.Textbox(label="Analysis Result")
            
            # Connect the action trigger to the processing loop
            text_button.click(fn=process_text, inputs=text_input, outputs=text_output)
            
        # --- TAB 2: COMPUTER VISION ---
        with gr.TabItem("🖼️ Computer Vision Hub"):
            gr.Markdown("### Image Classification via Vision Transformer (ViT)")
            image_input = gr.Image(type="pil", label="Upload Source Image")
            image_button = gr.Button("Classify Image", variant="primary")
            image_output = gr.Textbox(lines=4, label="Top Predicted Classes")
            
            image_button.click(fn=process_image, inputs=image_input, outputs=image_output)
            
        # --- TAB 3: AUDIO TRANSLATION ---
        with gr.TabItem("🔊 Speech-to-Text Transcriber"):
            gr.Markdown("### Automated Speech Recognition via Whisper")
            # This component allows users to upload pre-recorded files or speak directly into their microphone
            audio_input = gr.Audio(type="filepath", label="Record/Upload Audio Clip")
            audio_button = gr.Button("Transcribe Audio", variant="primary")
            audio_output = gr.Textbox(lines=4, label="Whisper Transcription Output")
            
            audio_button.click(fn=process_audio, inputs=audio_input, outputs=audio_output)

In [5]:
# ------------------------------------------------------------------
# Application Server Launch Execution
# ------------------------------------------------------------------
if __name__ == "__main__":
    # Launching with local server parameters enabled
    dashboard.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
